# 01 · WCA-Bench Data Exploration

This notebook explores the processed WCA-Bench dataset: table sizes, temporal coverage,
per-event volume, and the DNF rate.

**Prerequisites.** Run the pipeline first:

```bash
python scripts/download_data.py
python scripts/build_dataset.py --source raw   # or --source synthetic
```

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
PROCESSED = ROOT / "data" / "processed"
print("processed dir:", PROCESSED, "exists:", PROCESSED.exists())

## 1. Table inventory

Every Parquet table produced by the pipeline, with row and column counts.

In [ ]:
rows = []
for path in sorted(PROCESSED.glob("*.parquet")):
    df = pd.read_parquet(path)
    rows.append({"table": path.stem, "rows": len(df), "cols": df.shape[1]})

inventory = pd.DataFrame(rows).sort_values("rows", ascending=False)
inventory

## 2. Temporal coverage

Competition volume per year. WCA-Bench uses a **temporal** split, so the shape of this curve
matters more than the total row count.

In [ ]:
comps = pd.read_parquet(PROCESSED / "competitions.parquet")
date_col = next(c for c in ("start_date", "date", "year") if c in comps.columns)

comps["year"] = pd.to_datetime(comps[date_col], errors="coerce").dt.year
per_year = comps.groupby("year").size().rename("competitions").to_frame()
per_year.tail(25)

## 3. Records per event

Event volume is highly skewed: 3x3 dominates, while big-blind and multi-blind events are rare.
This is exactly why the evaluation protocol requires **per-event stratified reporting**.

In [ ]:
results = pd.read_parquet(PROCESSED / "results.parquet")
print("results rows:", len(results))
print("columns:", list(results.columns)[:20], "...")

if "event_id" in results.columns:
    by_event = (
        results.groupby("event_id")
        .size()
        .rename("records")
        .sort_values(ascending=False)
        .to_frame()
    )
    by_event["share"] = (by_event["records"] / by_event["records"].sum()).round(4)
    display(by_event)

## 4. Sentinel values and the DNF rate

`-1` = DNF, `-2` = DNS, `0` = no result. The global DNF rate is roughly 3%, but it is very uneven
across events, which motivates the hard-subset reporting in Task 3.

In [ ]:
best_col = next((c for c in ("best", "best_value") if c in results.columns), None)
if best_col:
    print(results[best_col].value_counts().head(5).rename("count"))
    dnf = (results[best_col] == -1).mean()
    dns = (results[best_col] == -2).mean()
    print(f"global DNF rate: {dnf:.4%}")
    print(f"global DNS rate: {dns:.4%}")

    if "event_id" in results.columns:
        per_event = (
            results.assign(is_dnf=results[best_col].eq(-1))
            .groupby("event_id")["is_dnf"]
            .agg(["mean", "size"])
            .rename(columns={"mean": "dnf_rate", "size": "records"})
            .sort_values("dnf_rate", ascending=False)
        )
        display(per_event)

## 5. Split sizes

The temporal split must partition the data exactly. The numbers below are also recorded in
`data/processed/manifest.json`.

In [ ]:
if "split" in results.columns:
    split_counts = results["split"].value_counts().rename("results").to_frame()
    split_counts["share"] = (split_counts["results"] / split_counts["results"].sum()).round(4)
    display(split_counts)
else:
    print("No 'split' column: run scripts/build_dataset.py first.")

## Next steps

- `02_domain_rules.ipynb` — verify the WCA-specific decodings (multi-blind, averages, sentinels).
- `03_baseline_analysis.ipynb` — read the released reports and compare baselines.